In [ ]:
# filename: gee_chengdu_ndvi_viirs_by_district.py
# Full name(s) of author(s): Shoshana Abikzer
# What this code does (1–2 lines): Uses the Google Earth Engine API to compute mean NDVI and mean VIIRS nightlights for Chengdu’s county-level admin units and exports a CSV.
# AI use disclosure (1–2 lines): Used for debugging; verified parameters, and ran/validated outputs.

import os
import sys
import json
import time
import random
import pathlib
import datetime as dt
from typing import Optional, Dict, Any, List

import ee


# ----------------------------
# 0) Config (edit these only)
# ----------------------------

# Your uploaded GADM table asset (you already have this in GEE)
GADM_ADM3_ASSET = "projects/project-goodnight-485705/assets/gadm41_CHN_3"

# Chengdu selection (GADM fields in your table)
FILTERS = {
    "COUNTRY": "China",
    "NAME_1": "Sichuan",
    "NAME_2": "Chengdu",
}

# Year (set per your project)
YEAR = 2023

# NDVI (MODIS)
MODIS_COLLECTION = "MODIS/061/MOD13Q1"   # 250m NDVI, 16-day
MODIS_NDVI_BAND = "NDVI"
MODIS_SCALE_FACTOR = 0.0001
MODIS_SCALE_M = 250

# VIIRS Nightlights (monthly)
VIIRS_COLLECTION = "NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG"  # stable monthly lights
VIIRS_RADIANCE_BAND = "avg_rad"
VIIRS_SCALE_M = 500

# Export
EXPORT_DESCRIPTION = f"Chengdu_NDVI_VIIRS_by_ADM3_{YEAR}"
EXPORT_FOLDER = "GEE_Exports"   # Google Drive folder (auto-created if absent)
EXPORT_FILEFORMAT = "CSV"

# ReduceRegions tiling
TILE_SCALE = 4


# ----------------------------
# 1) Helpers
# ----------------------------

def die(msg: str, code: int = 1) -> None:
    print(msg, file=sys.stderr)
    raise SystemExit(code)

def iso_date(y: int, m: int, d: int) -> str:
    return f"{y:04d}-{m:02d}-{d:02d}"

def init_ee() -> None:
    """
    Initializes Earth Engine. Works if:
      - You already authenticated locally (ee.Authenticate() done once), OR
      - Running in a GCP/Colab environment with EE credentials.
    """
    try:
        ee.Initialize()
    except Exception:
        try:
            ee.Authenticate()
            ee.Initialize()
        except Exception as e:
            die(
                "Earth Engine init failed. Run `earthengine authenticate` in a terminal, "
                "or call ee.Authenticate() in an interactive environment.\n"
                f"Error: {e}"
            )

def fc_from_asset(asset_id: str) -> ee.FeatureCollection:
    return ee.FeatureCollection(asset_id)

def apply_filters(fc: ee.FeatureCollection, filters: Dict[str, str]) -> ee.FeatureCollection:
    out = fc
    for k, v in filters.items():
        out = out.filter(ee.Filter.eq(k, v))
    return out

def modis_ndvi_mean(year: int) -> ee.Image:
    start = ee.Date(iso_date(year, 1, 1))
    end = ee.Date(iso_date(year + 1, 1, 1))
    col = (
        ee.ImageCollection(MODIS_COLLECTION)
        .filterDate(start, end)
        .select([MODIS_NDVI_BAND])
        .map(lambda img: img.multiply(MODIS_SCALE_FACTOR).copyProperties(img, img.propertyNames()))
    )
    return col.mean().rename("ndvi_mean")

def viirs_mean(year: int) -> ee.Image:
    start = ee.Date(iso_date(year, 1, 1))
    end = ee.Date(iso_date(year + 1, 1, 1))
    col = (
        ee.ImageCollection(VIIRS_COLLECTION)
        .filterDate(start, end)
        .select([VIIRS_RADIANCE_BAND])
    )
    return col.mean().rename("viirs_mean_avg_rad")

def add_city_labels(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    """
    Adds a few stable labels to make merges easier downstream.
    """
    def _f(feat: ee.Feature) -> ee.Feature:
        return (
            feat.set("city", "Chengdu")
                .set("province", FILTERS["NAME_1"])
                .set("country", FILTERS["COUNTRY"])
        )
    return fc.map(_f)

def reduce_to_units(
    units: ee.FeatureCollection,
    ndvi_img: ee.Image,
    viirs_img: ee.Image
) -> ee.FeatureCollection:
    """
    Computes mean NDVI and mean VIIRS for each ADM3 polygon.
    """
    combined = ee.Image.cat([ndvi_img, viirs_img])

    reduced = combined.reduceRegions(
        collection=units,
        reducer=ee.Reducer.mean(),
        scale=MODIS_SCALE_M,      # NDVI’s scale; VIIRS will be resampled internally
        tileScale=TILE_SCALE
    )

    # Clean up output field names to be explicit + add year
    def _cleanup(f: ee.Feature) -> ee.Feature:
        return (
            f.set("year", YEAR)
             .set("ndvi_mean", f.get("mean"))  # placeholder; will be overwritten below
        )

    # Earth Engine reducer returns 'mean' per band only if single band; with multi-band it returns
    # band-specific: 'ndvi_mean'/'viirs_mean_avg_rad' ONLY if the bands are named.
    # Since we named bands, the output should contain:
    #   - 'ndvi_mean' and 'viirs_mean_avg_rad'
    # We just ensure they exist and drop geometry-heavy props for smaller CSV.
    keep = [
        "year", "country", "province", "city",
        "GID_0", "GID_1", "GID_2", "GID_3",
        "NAME_0", "NAME_1", "NAME_2", "NAME_3",
        "VARNAME_3", "TYPE_3", "ENGTYPE_3",
        "ndvi_mean", "viirs_mean_avg_rad"
    ]

    def _select_props(f: ee.Feature) -> ee.Feature:
        props = ee.Dictionary.fromLists(
            ee.List(keep),
            ee.List(keep).map(lambda k: f.get(ee.String(k)))
        )
        return ee.Feature(None, props)

    return reduced.map(_select_props)

def export_table_to_drive(fc: ee.FeatureCollection) -> ee.batch.Task:
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=EXPORT_DESCRIPTION,
        folder=EXPORT_FOLDER,
        fileFormat=EXPORT_FILEFORMAT
    )
    task.start()
    return task

def monitor_task(task: ee.batch.Task, poll_s: int = 10, max_minutes: int = 60) -> None:
    """
    Prints status until completion or timeout.
    """
    started = time.time()
    deadline = started + max_minutes * 60
    last_state = None

    while True:
        status = task.status()
        state = status.get("state", "UNKNOWN")

        if state != last_state:
            print(json.dumps(status, indent=2, default=str))
            last_state = state

        if state in ("COMPLETED", "FAILED", "CANCELLED"):
            if state != "COMPLETED":
                die(f"Export did not complete. Final state: {state}\nStatus: {json.dumps(status, indent=2)}")
            print("Export completed.")
            return

        if time.time() > deadline:
            die(f"Timed out waiting for task after {max_minutes} minutes. Check GEE Tasks/Drive manually.")

        time.sleep(poll_s)


# ----------------------------
# 2) Main
# ----------------------------

def main() -> None:
    init_ee()

    adm3 = fc_from_asset(GADM_ADM3_ASSET)
    chengdu_units = apply_filters(adm3, FILTERS)

    # Basic sanity: count should be > 0
    n = chengdu_units.size().getInfo()
    if not n or n == 0:
        die(
            "No features matched your Chengdu filters. "
            "Double-check field names/values in your GADM asset."
        )
    print(f"Matched Chengdu ADM3 units: {n}")

    # Add stable labels
    chengdu_units = add_city_labels(chengdu_units)

    # Build annual composites
    ndvi_img = modis_ndvi_mean(YEAR)
    viirs_img = viirs_mean(YEAR)

    # Reduce
    stats = reduce_to_units(chengdu_units, ndvi_img, viirs_img)

    # (Optional) print a small sample to confirm fields exist
    sample = stats.limit(5).getInfo()
    print("Sample rows (first 5):")
    print(json.dumps(sample, indent=2))

    # Export
    task = export_table_to_drive(stats)
    print("Started export task:", task.id)

    # Monitor (optional; safe to leave running without this)
    monitor_task(task, poll_s=10, max_minutes=60)


if __name__ == "__main__":
    main()


In [ ]:
# filename: scrape_admin_unit_names_incomplete.py
# Full name(s) of author(s): Shoshana Abikzer, Rebecca [LASTNAME]
# What this code does (1–2 lines): In-progress web scraping pipeline to collect Chengdu administrative unit names from Wikipedia-like HTML tables for later joins/validation.
# AI use disclosure (1–2 lines): Drafted with AI assistance for parser structure; author will complete selectors + validate extracted units.

import re
import time
import json
import random
from dataclasses import dataclass
from typing import Optional, List, Dict, Any, Tuple

import requests
from bs4 import BeautifulSoup


# ----------------------------
# 0) Config
# ----------------------------

SEED_URLS = [
    # TODO: confirm final source URL(s) used by the group before submission
    # Example placeholders:
    "https://en.wikipedia.org/wiki/Chengdu",
    "https://en.wikipedia.org/wiki/List_of_administrative_divisions_of_Sichuan",
]

USER_AGENT = "MACS30112-GoodNight/1.0 (course project; contact: <your-email>)"
HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Language": "en-US,en;q=0.9",
}
TIMEOUT_S = 30
MIN_DELAY_S = 1.0
MAX_DELAY_S = 3.0


# ----------------------------
# 1) Helpers
# ----------------------------

def polite_sleep():
    time.sleep(random.uniform(MIN_DELAY_S, MAX_DELAY_S))

def get(url: str) -> requests.Response:
    r = requests.get(url, headers=HEADERS, timeout=TIMEOUT_S)
    r.raise_for_status()
    return r

def normalize_name(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    s = s.replace("\xa0", " ")
    return s

def extract_tables(html: str) -> List[BeautifulSoup]:
    soup = BeautifulSoup(html, "html.parser")
    # TODO: refine selection criteria per the target page(s)
    return soup.select("table.wikitable")

def parse_units_from_table(table: BeautifulSoup) -> List[Dict[str, Any]]:
    """
    TODO: implement page-specific parsing:
      - Identify header row
      - Determine columns for: Chinese name, English name, admin type
      - Handle footnotes/sup tags
      - Return consistent dict schema
    """
    units: List[Dict[str, Any]] = []

    rows = table.select("tr")
    if not rows:
        return units

    # TODO: inspect header structure and map column indices
    # For now, collect raw text as a stub.
    for r in rows[1:]:
        cells = r.select("th, td")
        if not cells:
            continue
        raw = [normalize_name(c.get_text(" ", strip=True)) for c in cells]
        units.append({"raw_cells": raw})

    return units


# ----------------------------
# 2) Main (incomplete)
# ----------------------------

def main() -> None:
    all_units: List[Dict[str, Any]] = []

    for url in SEED_URLS:
        resp = get(url)
        polite_sleep()
        tables = extract_tables(resp.text)

        for t in tables:
            units = parse_units_from_table(t)
            # TODO: add filtering logic so only Chengdu-specific admin units are kept
            all_units.extend(units)

    # TODO: deduplicate + standardize fields (english_name, chinese_name, admin_type)
    out = {
        "source_urls": SEED_URLS,
        "n_rows_raw": len(all_units),
        "rows": all_units[:200],  # keep small while developing
    }

    with open("chengdu_units_scrape_incomplete.json", "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    print("Wrote: chengdu_units_scrape_incomplete.json")


if __name__ == "__main__":
    main()
